<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/11_Final_Publication_Results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 11.0 — ENVIRONMENT & ARTIFACT DISCOVERY
# ============================================================

from pathlib import Path
import json
import hashlib
import warnings
import gc
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 11")
print("FINAL PUBLICATION RESULTS")
print("=" * 100)


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

try:
    from google.colab import drive

    DRIVE_ROOT = Path("/content/drive")

    if not DRIVE_ROOT.exists():
        DRIVE_ROOT.mkdir(
            parents=True,
            exist_ok=True
        )

    try:
        drive.mount(
            str(DRIVE_ROOT),
            force_remount=False
        )
    except Exception:
        pass

except Exception:
    DRIVE_ROOT = Path("/content/drive")


# ------------------------------------------------------------
# Project root discovery
# ------------------------------------------------------------

PROJECT_CANDIDATES = [

    Path("/content/drive/MyDrive/AIR_LLM_Research"),

    Path("/content/air_llm_drive/MyDrive/AIR_LLM_Research"),

    Path("/content/air_llm_drive_03/MyDrive/AIR_LLM_Research"),

]


PROJECT_ROOT = None

for candidate in PROJECT_CANDIDATES:

    try:

        if candidate.is_dir():

            PROJECT_ROOT = candidate
            break

    except Exception:
        continue


if PROJECT_ROOT is None:

    raise FileNotFoundError(
        "AIR_LLM_Research project could not be located.\n"
        "Expected location:\n"
        "/content/drive/MyDrive/AIR_LLM_Research"
    )


print("\nPROJECT ROOT")
print("-" * 100)
print(PROJECT_ROOT)


# ------------------------------------------------------------
# Notebook directories
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

NB03_DIR = DATA_DIR / "notebook_03"
NB04_DIR = DATA_DIR / "notebook_04"
NB05_DIR = DATA_DIR / "notebook_05"
NB06_DIR = DATA_DIR / "notebook_06"
NB07_DIR = DATA_DIR / "notebook_07"
NB08_DIR = DATA_DIR / "notebook_08"
NB09_DIR = DATA_DIR / "notebook_09"
NB10_DIR = DATA_DIR / "notebook_10"

NB11_DIR = DATA_DIR / "notebook_11"


# ------------------------------------------------------------
# Publication directories
# ------------------------------------------------------------

PUBLICATION_DIR = NB11_DIR / "publication"

TABLE_DIR = PUBLICATION_DIR / "tables"
FIGURE_DIR = PUBLICATION_DIR / "figures"
SUMMARY_DIR = PUBLICATION_DIR / "summary"
STATISTICS_DIR = PUBLICATION_DIR / "statistics"
MANIFEST_DIR = PUBLICATION_DIR / "metadata"


for directory in [
    NB11_DIR,
    PUBLICATION_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    SUMMARY_DIR,
    STATISTICS_DIR,
    MANIFEST_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("\nOUTPUT DIRECTORY")
print("-" * 100)
print(PUBLICATION_DIR)

In [ ]:
# ============================================================
# NOTEBOOK 11.0 — ENVIRONMENT & ARTIFACT DISCOVERY
# ============================================================

from pathlib import Path
import json
import hashlib
import warnings
import gc
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 11")
print("FINAL PUBLICATION RESULTS")
print("=" * 100)


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

try:
    from google.colab import drive

    DRIVE_ROOT = Path("/content/drive")

    if not DRIVE_ROOT.exists():
        DRIVE_ROOT.mkdir(
            parents=True,
            exist_ok=True
        )

    try:
        drive.mount(
            str(DRIVE_ROOT),
            force_remount=False
        )
    except Exception:
        pass

except Exception:
    DRIVE_ROOT = Path("/content/drive")


# ------------------------------------------------------------
# Project root discovery
# ------------------------------------------------------------

PROJECT_CANDIDATES = [

    Path("/content/drive/MyDrive/AIR_LLM_Research"),

    Path("/content/air_llm_drive/MyDrive/AIR_LLM_Research"),

    Path("/content/air_llm_drive_03/MyDrive/AIR_LLM_Research"),

]


PROJECT_ROOT = None

for candidate in PROJECT_CANDIDATES:

    try:

        if candidate.is_dir():

            PROJECT_ROOT = candidate
            break

    except Exception:
        continue


if PROJECT_ROOT is None:

    raise FileNotFoundError(
        "AIR_LLM_Research project could not be located.\n"
        "Expected location:\n"
        "/content/drive/MyDrive/AIR_LLM_Research"
    )


print("\nPROJECT ROOT")
print("-" * 100)
print(PROJECT_ROOT)


# ------------------------------------------------------------
# Notebook directories
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

NB03_DIR = DATA_DIR / "notebook_03"
NB04_DIR = DATA_DIR / "notebook_04"
NB05_DIR = DATA_DIR / "notebook_05"
NB06_DIR = DATA_DIR / "notebook_06"
NB07_DIR = DATA_DIR / "notebook_07"
NB08_DIR = DATA_DIR / "notebook_08"
NB09_DIR = DATA_DIR / "notebook_09"
NB10_DIR = DATA_DIR / "notebook_10"

NB11_DIR = DATA_DIR / "notebook_11"


# ------------------------------------------------------------
# Publication directories
# ------------------------------------------------------------

PUBLICATION_DIR = NB11_DIR / "publication"

TABLE_DIR = PUBLICATION_DIR / "tables"
FIGURE_DIR = PUBLICATION_DIR / "figures"
SUMMARY_DIR = PUBLICATION_DIR / "summary"
STATISTICS_DIR = PUBLICATION_DIR / "statistics"
MANIFEST_DIR = PUBLICATION_DIR / "metadata"


for directory in [
    NB11_DIR,
    PUBLICATION_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    SUMMARY_DIR,
    STATISTICS_DIR,
    MANIFEST_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("\nOUTPUT DIRECTORY")
print("-" * 100)
print(PUBLICATION_DIR)

In [ ]:
# ============================================================
# CELL 11.1 — LOAD RESEARCH CONFIGURATION
# ============================================================

DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

MECHANISMS = [
    "MCAR",
    "MAR",
    "MNAR_APPROXIMATION"
]

MISSINGNESS_RATES = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

REPETITIONS = 5

MASTER_SEED = 42


RUN_TIMESTAMP = datetime.now(
    timezone.utc
).isoformat()


print("=" * 100)
print("AIR-LLM RESEARCH CONFIGURATION")
print("=" * 100)

print(f"Datasets          : {len(DATASETS)}")
print(f"Mechanisms        : {MECHANISMS}")
print(f"Missingness rates : {MISSINGNESS_RATES}")
print(f"Repetitions       : {REPETITIONS}")
print(f"Master seed       : {MASTER_SEED}")
print(f"Timestamp         : {RUN_TIMESTAMP}")

In [ ]:
# ============================================================
# CELL 11.2 — LOCATE NOTEBOOK OUTPUTS
# ============================================================

def find_csv_files(root):

    if not root.exists():
        return []

    return sorted(
        root.rglob("*.csv")
    )


def find_json_files(root):

    if not root.exists():
        return []

    return sorted(
        root.rglob("*.json")
    )


NOTEBOOK_OUTPUTS = {}

for number, directory in {
    "03": NB03_DIR,
    "04": NB04_DIR,
    "05": NB05_DIR,
    "06": NB06_DIR,
    "07": NB07_DIR,
    "08": NB08_DIR,
    "09": NB09_DIR,
    "10": NB10_DIR
}.items():

    NOTEBOOK_OUTPUTS[number] = {
        "csv": find_csv_files(directory),
        "json": find_json_files(directory)
    }


print("=" * 100)
print("AIR-LLM — NOTEBOOK OUTPUT DISCOVERY")
print("=" * 100)

for number, outputs in NOTEBOOK_OUTPUTS.items():

    print(
        f"\nNotebook {number}"
    )

    print(
        f"  CSV files  : {len(outputs['csv'])}"
    )

    print(
        f"  JSON files : {len(outputs['json'])}"
    )

In [ ]:
# ============================================================
# CELL 11.3 — LOAD EXPERIMENTAL RESULTS
# ============================================================

def load_all_csvs(root):

    records = []

    for path in find_csv_files(root):

        try:

            df = pd.read_csv(
                path,
                low_memory=False
            )

            if len(df) == 0:
                continue

            records.append({
                "path": path,
                "name": path.name,
                "relative": str(
                    path.relative_to(root)
                ),
                "data": df
            })

        except Exception as exc:

            print(
                f"WARNING: Could not read {path.name}: {exc}"
            )

    return records


RESULT_FILES = {}

for number, directory in {
    "03": NB03_DIR,
    "04": NB04_DIR,
    "05": NB05_DIR,
    "06": NB06_DIR,
    "07": NB07_DIR,
    "08": NB08_DIR,
    "09": NB09_DIR,
    "10": NB10_DIR
}.items():

    RESULT_FILES[number] = load_all_csvs(
        directory
    )


print("=" * 100)
print("RESULT FILE INVENTORY")
print("=" * 100)

for number, records in RESULT_FILES.items():

    print(
        f"Notebook {number}: "
        f"{len(records)} CSV files"
    )

    for record in records[:10]:

        print(
            f"  - {record['relative']}"
        )

In [ ]:
# ============================================================
# CELL 11.5 — RESULT SCHEMA INSPECTION
# ============================================================

def inspect_table(
    name,
    df
):

    print("\n" + "-" * 100)
    print(name)

    if df is None or df.empty:

        print("EMPTY / NOT FOUND")
        return

    print(
        f"Rows    : {len(df):,}"
    )

    print(
        f"Columns : {len(df.columns)}"
    )

    print(
        "Columns :"
    )

    for column in df.columns:

        print(
            f"  - {column}"
        )


inspect_table(
    "UTILITY RESULTS",
    UTILITY_RESULTS
)

inspect_table(
    "CONFIDENCE RESULTS",
    CONFIDENCE_RESULTS
)

inspect_table(
    "EXPLANATION RESULTS",
    EXPLANATION_RESULTS
)

inspect_table(
    "ROBUSTNESS RESULTS",
    ROBUSTNESS_RESULTS
)

In [ ]:
# ============================================================
# CELL 11.5 — RESULT SCHEMA INSPECTION
# ============================================================

def inspect_table(
    name,
    df
):

    print("\n" + "-" * 100)
    print(name)

    if df is None or df.empty:

        print("EMPTY / NOT FOUND")
        return

    print(
        f"Rows    : {len(df):,}"
    )

    print(
        f"Columns : {len(df.columns)}"
    )

    print(
        "Columns :"
    )

    for column in df.columns:

        print(
            f"  - {column}"
        )


inspect_table(
    "UTILITY RESULTS",
    UTILITY_RESULTS
)

inspect_table(
    "CONFIDENCE RESULTS",
    CONFIDENCE_RESULTS
)

inspect_table(
    "EXPLANATION RESULTS",
    EXPLANATION_RESULTS
)

inspect_table(
    "ROBUSTNESS RESULTS",
    ROBUSTNESS_RESULTS
)

In [ ]:
# ============================================================
# CELL 11.7 — FINAL DATASET / FEATURE COVERAGE
# ============================================================

COVERAGE_ROWS = []

for dataset_id in DATASETS:

    row = {
        "dataset_id": dataset_id,
        "target": TARGET_REGISTRY[dataset_id]
    }

    if not UTILITY_RESULTS.empty:

        subset = UTILITY_RESULTS[
            UTILITY_RESULTS.get(
                "dataset_id",
                pd.Series(
                    dtype=object
                )
            ) == dataset_id
        ]

        row["utility_rows"] = len(
            subset
        )

        if "feature" in subset.columns:

            row["features_evaluated"] = (
                subset["feature"]
                .nunique()
            )

        else:

            row["features_evaluated"] = np.nan

    else:

        row["utility_rows"] = 0
        row["features_evaluated"] = np.nan

    COVERAGE_ROWS.append(row)


COVERAGE_DF = pd.DataFrame(
    COVERAGE_ROWS
)


display(
    COVERAGE_DF
)


COVERAGE_DF.to_csv(
    TABLE_DIR / "dataset_feature_coverage.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.7 — FINAL DATASET / FEATURE COVERAGE
# ============================================================

COVERAGE_ROWS = []

for dataset_id in DATASETS:

    row = {
        "dataset_id": dataset_id,
        "target": TARGET_REGISTRY[dataset_id]
    }

    if not UTILITY_RESULTS.empty:

        subset = UTILITY_RESULTS[
            UTILITY_RESULTS.get(
                "dataset_id",
                pd.Series(
                    dtype=object
                )
            ) == dataset_id
        ]

        row["utility_rows"] = len(
            subset
        )

        if "feature" in subset.columns:

            row["features_evaluated"] = (
                subset["feature"]
                .nunique()
            )

        else:

            row["features_evaluated"] = np.nan

    else:

        row["utility_rows"] = 0
        row["features_evaluated"] = np.nan

    COVERAGE_ROWS.append(row)


COVERAGE_DF = pd.DataFrame(
    COVERAGE_ROWS
)


display(
    COVERAGE_DF
)


COVERAGE_DF.to_csv(
    TABLE_DIR / "dataset_feature_coverage.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.8 — FINAL STRATEGY SELECTION RESULTS
# ============================================================

if UTILITY_RESULTS.empty:

    FINAL_SELECTION_DF = pd.DataFrame()

    print(
        "Utility result table was not automatically identified."
    )

else:

    FINAL_SELECTION_DF = UTILITY_RESULTS.copy()

    # Identify selected rows when possible.

    if "selected" in FINAL_SELECTION_DF.columns:

        values = (
            FINAL_SELECTION_DF["selected"]
            .astype(str)
            .str.lower()
        )

        FINAL_SELECTION_DF = (
            FINAL_SELECTION_DF[
                values.isin(
                    [
                        "true",
                        "1",
                        "yes",
                        "selected"
                    ]
                )
                .copy()
            )

    elif "rank" in FINAL_SELECTION_DF.columns:

        FINAL_SELECTION_DF = (
            FINAL_SELECTION_DF[
                pd.to_numeric(
                    FINAL_SELECTION_DF["rank"],
                    errors="coerce"
                ) == 1
            ]
            .copy()
        )


print("=" * 100)
print("FINAL STRATEGY SELECTION")
print("=" * 100)

print(
    f"Selected records : "
    f"{len(FINAL_SELECTION_DF):,}"
)

display(
    FINAL_SELECTION_DF.head(20)
)


FINAL_SELECTION_DF.to_csv(
    TABLE_DIR / "final_strategy_selection.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.9 — STRATEGY SELECTION FREQUENCY
# ============================================================

if (
    not FINAL_SELECTION_DF.empty
    and "strategy" in FINAL_SELECTION_DF.columns
):

    STRATEGY_FREQUENCY = (
        FINAL_SELECTION_DF
        ["strategy"]
        .value_counts()
        .rename_axis("strategy")
        .reset_index(
            name="selection_count"
        )
    )

    STRATEGY_FREQUENCY[
        "selection_percentage"
    ] = (
        STRATEGY_FREQUENCY[
            "selection_count"
        ]
        /
        STRATEGY_FREQUENCY[
            "selection_count"
        ].sum()
        * 100
    )

else:

    STRATEGY_FREQUENCY = pd.DataFrame(
        columns=[
            "strategy",
            "selection_count",
            "selection_percentage"
        ]
    )


display(
    STRATEGY_FREQUENCY
)


STRATEGY_FREQUENCY.to_csv(
    TABLE_DIR / "strategy_selection_frequency.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.10 — DATASET × STRATEGY SUMMARY
# ============================================================

if (
    not FINAL_SELECTION_DF.empty
    and "dataset_id" in FINAL_SELECTION_DF.columns
    and "strategy" in FINAL_SELECTION_DF.columns
):

    DATASET_STRATEGY_SUMMARY = (
        FINAL_SELECTION_DF
        .groupby(
            [
                "dataset_id",
                "strategy"
            ],
            dropna=False
        )
        .size()
        .reset_index(
            name="selection_count"
        )
    )

else:

    DATASET_STRATEGY_SUMMARY = pd.DataFrame()


display(
    DATASET_STRATEGY_SUMMARY
)


DATASET_STRATEGY_SUMMARY.to_csv(
    TABLE_DIR / "dataset_strategy_summary.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.11 — UTILITY SUMMARY
# ============================================================

if (
    not UTILITY_RESULTS.empty
    and "utility" in UTILITY_RESULTS.columns
):

    UTILITY_RESULTS["utility"] = pd.to_numeric(
        UTILITY_RESULTS["utility"],
        errors="coerce"
    )

    UTILITY_SUMMARY = (
        UTILITY_RESULTS
        .groupby(
            [
                c for c in [
                    "dataset_id",
                    "strategy"
                ]
                if c in UTILITY_RESULTS.columns
            ],
            dropna=False
        )["utility"]
        .agg(
            [
                "count",
                "mean",
                "std",
                "median",
                "min",
                "max"
            ]
        )
        .reset_index()
    )

else:

    UTILITY_SUMMARY = pd.DataFrame()


display(
    UTILITY_SUMMARY
)


UTILITY_SUMMARY.to_csv(
    TABLE_DIR / "utility_summary.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.12 — CONFIDENCE AND ABSTENTION SUMMARY
# ============================================================

if CONFIDENCE_RESULTS.empty:

    CONFIDENCE_SUMMARY = pd.DataFrame()

else:

    confidence = CONFIDENCE_RESULTS.copy()

    if "confidence" in confidence.columns:

        confidence["confidence"] = pd.to_numeric(
            confidence["confidence"],
            errors="coerce"
        )

    if "abstention" in confidence.columns:

        confidence["abstention_binary"] = (
            confidence["abstention"]
            .astype(str)
            .str.lower()
            .isin(
                [
                    "true",
                    "1",
                    "yes"
                ]
            )
            .astype(int)
        )

    group_columns = [
        c for c in [
            "dataset_id",
            "mechanism"
        ]
        if c in confidence.columns
    ]

    if group_columns:

        aggregations = {}

        if "confidence" in confidence.columns:
            aggregations["confidence"] = [
                "count",
                "mean",
                "std",
                "median"
            ]

        if "abstention_binary" in confidence.columns:
            aggregations[
                "abstention_binary"
            ] = [
                "mean",
                "sum"
            ]

        CONFIDENCE_SUMMARY = (
            confidence
            .groupby(
                group_columns,
                dropna=False
            )
            .agg(
                aggregations
            )
            .reset_index()
        )

        CONFIDENCE_SUMMARY.columns = [
            "_".join(
                [str(x) for x in col if str(x) != ""]
            ).strip("_")
            if isinstance(col, tuple)
            else str(col)
            for col in CONFIDENCE_SUMMARY.columns
        ]

    else:

        CONFIDENCE_SUMMARY = pd.DataFrame()


display(
    CONFIDENCE_SUMMARY
)


CONFIDENCE_SUMMARY.to_csv(
    TABLE_DIR / "confidence_abstention_summary.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.13 — CROSS-DATASET SUMMARY
# ============================================================

CROSS_DATASET_ROWS = []

for dataset_id in DATASETS:

    row = {
        "dataset_id": dataset_id
    }

    if (
        not FINAL_SELECTION_DF.empty
        and "dataset_id" in FINAL_SELECTION_DF.columns
    ):

        subset = FINAL_SELECTION_DF[
            FINAL_SELECTION_DF[
                "dataset_id"
            ] == dataset_id
        ]

        row["selected_features"] = len(
            subset
        )

        if "strategy" in subset.columns:

            row["unique_strategies"] = (
                subset["strategy"]
                .nunique()
            )

            if len(subset):

                row["dominant_strategy"] = (
                    subset["strategy"]
                    .value_counts()
                    .index[0]
                )

            else:

                row["dominant_strategy"] = np.nan

        else:

            row["unique_strategies"] = np.nan
            row["dominant_strategy"] = np.nan

    else:

        row["selected_features"] = 0
        row["unique_strategies"] = np.nan
        row["dominant_strategy"] = np.nan

    CROSS_DATASET_ROWS.append(row)


CROSS_DATASET_SUMMARY = pd.DataFrame(
    CROSS_DATASET_ROWS
)


display(
    CROSS_DATASET_SUMMARY
)


CROSS_DATASET_SUMMARY.to_csv(
    TABLE_DIR / "cross_dataset_summary.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.14 — FINAL STATISTICAL ANALYSIS
# ============================================================

from scipy import stats


STATISTICAL_RESULTS = []


if (
    not UTILITY_RESULTS.empty
    and "strategy" in UTILITY_RESULTS.columns
    and "utility" in UTILITY_RESULTS.columns
):

    temp = UTILITY_RESULTS.copy()

    temp["utility"] = pd.to_numeric(
        temp["utility"],
        errors="coerce"
    )

    temp = temp.dropna(
        subset=["utility"]
    )

    strategies = (
        temp["strategy"]
        .dropna()
        .unique()
        .tolist()
    )

    # --------------------------------------------------------
    # Pairwise Welch comparisons
    # --------------------------------------------------------

    for i in range(len(strategies)):

        for j in range(i + 1, len(strategies)):

            strategy_a = strategies[i]
            strategy_b = strategies[j]

            a = temp[
                temp["strategy"] == strategy_a
            ]["utility"].to_numpy()

            b = temp[
                temp["strategy"] == strategy_b
            ]["utility"].to_numpy()

            if len(a) < 2 or len(b) < 2:
                continue

            test = stats.ttest_ind(
                a,
                b,
                equal_var=False,
                nan_policy="omit"
            )

            STATISTICAL_RESULTS.append({

                "strategy_a": strategy_a,

                "strategy_b": strategy_b,

                "n_a": len(a),

                "n_b": len(b),

                "mean_a": np.mean(a),

                "mean_b": np.mean(b),

                "mean_difference":
                    np.mean(a) - np.mean(b),

                "t_statistic":
                    test.statistic,

                "p_value":
                    test.pvalue
            })


STATISTICAL_DF = pd.DataFrame(
    STATISTICAL_RESULTS
)


if not STATISTICAL_DF.empty:

    STATISTICAL_DF[
        "p_adjusted_bonferroni"
    ] = (
        STATISTICAL_DF["p_value"]
        *
        len(STATISTICAL_DF)
    ).clip(
        upper=1.0
    )

    STATISTICAL_DF[
        "significant_05"
    ] = (
        STATISTICAL_DF[
            "p_adjusted_bonferroni"
        ] < 0.05
    )


display(
    STATISTICAL_DF.head(20)
)


STATISTICAL_DF.to_csv(
    STATISTICS_DIR / "final_pairwise_statistics.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.15 — EFFECT SIZE ANALYSIS
# ============================================================

EFFECT_ROWS = []


if not STATISTICAL_DF.empty:

    for _, row in STATISTICAL_DF.iterrows():

        a = UTILITY_RESULTS[
            UTILITY_RESULTS["strategy"]
            == row["strategy_a"]
        ]["utility"].dropna()

        b = UTILITY_RESULTS[
            UTILITY_RESULTS["strategy"]
            == row["strategy_b"]
        ]["utility"].dropna()

        if len(a) < 2 or len(b) < 2:
            continue

        pooled_sd = np.sqrt(
            (
                (len(a) - 1) * a.var(ddof=1)
                +
                (len(b) - 1) * b.var(ddof=1)
            )
            /
            (
                len(a) + len(b) - 2
            )
        )

        if pooled_sd == 0:
            cohens_d = np.nan
        else:
            cohens_d = (
                a.mean() - b.mean()
            ) / pooled_sd

        EFFECT_ROWS.append({

            "strategy_a":
                row["strategy_a"],

            "strategy_b":
                row["strategy_b"],

            "cohens_d":
                cohens_d,

            "absolute_effect":
                abs(cohens_d)
                if pd.notna(cohens_d)
                else np.nan
        })


EFFECT_SIZE_DF = pd.DataFrame(
    EFFECT_ROWS
)


if not EFFECT_SIZE_DF.empty:

    def effect_interpretation(d):

        if pd.isna(d):
            return "not_estimable"

        d = abs(d)

        if d < 0.20:
            return "negligible"

        if d < 0.50:
            return "small"

        if d < 0.80:
            return "medium"

        return "large"

    EFFECT_SIZE_DF[
        "interpretation"
    ] = EFFECT_SIZE_DF[
        "cohens_d"
    ].apply(
        effect_interpretation
    )


display(
    EFFECT_SIZE_DF.head(20)
)


EFFECT_SIZE_DF.to_csv(
    STATISTICS_DIR / "effect_sizes.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 11.16 — PUBLICATION TABLE 1
# FINAL STRATEGY SELECTIONS
# ============================================================

if not FINAL_SELECTION_DF.empty:

    publication_columns = [
        c for c in [
            "dataset_id",
            "feature",
            "strategy",
            "mechanism",
            "rate",
            "utility",
            "confidence",
            "abstention"
        ]
        if c in FINAL_SELECTION_DF.columns
    ]

    TABLE_1 = FINAL_SELECTION_DF[
        publication_columns
    ].copy()

else:

    TABLE_1 = pd.DataFrame()


TABLE_1.to_csv(
    TABLE_DIR / "Table_1_Final_Strategy_Selections.csv",
    index=False
)

display(
    TABLE_1.head(30)
)

In [ ]:
# ============================================================
# CELL 11.17 — PUBLICATION TABLE 2
# STRATEGY DISTRIBUTION
# ============================================================

TABLE_2 = STRATEGY_FREQUENCY.copy()

if not TABLE_2.empty:

    TABLE_2[
        "selection_percentage"
    ] = TABLE_2[
        "selection_percentage"
    ].round(2)


TABLE_2.to_csv(
    TABLE_DIR / "Table_2_Strategy_Distribution.csv",
    index=False
)

display(
    TABLE_2
)

In [ ]:
# ============================================================
# CELL 11.18 — PUBLICATION TABLE 3
# DATASET-LEVEL RESULTS
# ============================================================

TABLE_3 = CROSS_DATASET_SUMMARY.copy()


TABLE_3.to_csv(
    TABLE_DIR / "Table_3_Cross_Dataset_Results.csv",
    index=False
)


display(
    TABLE_3
)

In [ ]:
# ============================================================
# CELL 11.19 — PUBLICATION TABLE 4
# STATISTICAL RESULTS
# ============================================================

if not STATISTICAL_DF.empty:

    TABLE_4 = STATISTICAL_DF.copy()

    numeric_columns = [
        c for c in [
            "mean_a",
            "mean_b",
            "mean_difference",
            "t_statistic",
            "p_value",
            "p_adjusted_bonferroni"
        ]
        if c in TABLE_4.columns
    ]

    TABLE_4[numeric_columns] = (
        TABLE_4[numeric_columns]
        .round(6)
    )

else:

    TABLE_4 = pd.DataFrame()


TABLE_4.to_csv(
    TABLE_DIR / "Table_4_Statistical_Analysis.csv",
    index=False
)

display(
    TABLE_4.head(30)
)

In [ ]:
# ============================================================
# CELL 11.20 — FIGURE 1
# STRATEGY SELECTION FREQUENCY
# ============================================================

import matplotlib.pyplot as plt


if not STRATEGY_FREQUENCY.empty:

    plt.figure(
        figsize=(10, 6)
    )

    plt.bar(
        STRATEGY_FREQUENCY["strategy"].astype(str),
        STRATEGY_FREQUENCY[
            "selection_count"
        ]
    )

    plt.xlabel(
        "Selected Imputation Strategy"
    )

    plt.ylabel(
        "Number of Selections"
    )

    plt.title(
        "AIR-LLM Strategy Selection Frequency"
    )

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR
        / "Figure_1_Strategy_Selection_Frequency.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# ============================================================
# CELL 11.21 — FIGURE 2
# UTILITY DISTRIBUTION
# ============================================================

if (
    not UTILITY_RESULTS.empty
    and "utility" in UTILITY_RESULTS.columns
    and "strategy" in UTILITY_RESULTS.columns
):

    plot_df = UTILITY_RESULTS[
        [
            "strategy",
            "utility"
        ]
    ].dropna()

    strategies = (
        plot_df["strategy"]
        .unique()
        .tolist()
    )

    data = [
        plot_df.loc[
            plot_df["strategy"] == strategy,
            "utility"
        ].to_numpy()
        for strategy in strategies
    ]

    if data:

        plt.figure(
            figsize=(12, 6)
        )

        plt.boxplot(
            data,
            labels=strategies,
            showfliers=False
        )

        plt.xlabel(
            "Strategy"
        )

        plt.ylabel(
            "Utility"
        )

        plt.title(
            "AIR-LLM Utility Distribution Across Candidate Strategies"
        )

        plt.xticks(
            rotation=45,
            ha="right"
        )

        plt.tight_layout()

        plt.savefig(
            FIGURE_DIR
            / "Figure_2_Utility_Distribution.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

In [ ]:
# ============================================================
# CELL 11.22 — FIGURE 3
# DATASET-LEVEL UTILITY
# ============================================================

if (
    not UTILITY_RESULTS.empty
    and "dataset_id" in UTILITY_RESULTS.columns
    and "utility" in UTILITY_RESULTS.columns
):

    dataset_utility = (
        UTILITY_RESULTS
        .groupby(
            "dataset_id"
        )["utility"]
        .mean()
        .reset_index()
    )

    plt.figure(
        figsize=(9, 6)
    )

    plt.bar(
        dataset_utility["dataset_id"],
        dataset_utility["utility"]
    )

    plt.xlabel(
        "Dataset"
    )

    plt.ylabel(
        "Mean Utility"
    )

    plt.title(
        "AIR-LLM Mean Utility Across Datasets"
    )

    plt.xticks(
        rotation=30,
        ha="right"
    )

    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR
        / "Figure_3_Cross_Dataset_Utility.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# ============================================================
# CELL 11.23 — FIGURE 4
# CONFIDENCE DISTRIBUTION
# ============================================================

if (
    not CONFIDENCE_RESULTS.empty
    and "confidence" in CONFIDENCE_RESULTS.columns
):

    confidence_values = pd.to_numeric(
        CONFIDENCE_RESULTS["confidence"],
        errors="coerce"
    ).dropna()

    if len(confidence_values):

        plt.figure(
            figsize=(9, 6)
        )

        plt.hist(
            confidence_values,
            bins=20
        )

        plt.xlabel(
            "AIR-LLM Confidence"
        )

        plt.ylabel(
            "Frequency"
        )

        plt.title(
            "Distribution of AIR-LLM Selection Confidence"
        )

        plt.tight_layout()

        plt.savefig(
            FIGURE_DIR
            / "Figure_4_Confidence_Distribution.png",
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

In [ ]:
# ============================================================
# CELL 11.24 — FINAL AIR-LLM RESEARCH SUMMARY
# ============================================================

SUMMARY = {

    "framework": "AIR-LLM",

    "full_name":
        "Adaptive Missing-Value Imputation via "
        "LLM-Guided Recommendation and "
        "Empirical Strategy Selection",

    "datasets":
        DATASETS,

    "number_of_datasets":
        len(DATASETS),

    "mechanisms":
        MECHANISMS,

    "missingness_rates":
        MISSINGNESS_RATES,

    "repetitions":
        REPETITIONS,

    "master_seed":
        MASTER_SEED,

    "selected_features":
        int(
            len(FINAL_SELECTION_DF)
        ),

    "unique_selected_strategies":
        int(
            FINAL_SELECTION_DF[
                "strategy"
            ].nunique()
        )
        if (
            not FINAL_SELECTION_DF.empty
            and "strategy" in FINAL_SELECTION_DF.columns
        )
        else 0,

    "statistical_comparisons":
        int(
            len(STATISTICAL_DF)
        ),

    "effect_size_comparisons":
        int(
            len(EFFECT_SIZE_DF)
        ),

    "generated_utc":
        RUN_TIMESTAMP
}


SUMMARY_PATH = (
    SUMMARY_DIR
    / "AIR_LLM_final_research_summary.json"
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        SUMMARY,
        f,
        indent=2,
        default=str
    )


print("=" * 100)
print("AIR-LLM FINAL RESEARCH SUMMARY")
print("=" * 100)

for key, value in SUMMARY.items():

    print(
        f"{key:35s}: {value}"
    )

In [ ]:
# ============================================================
# CELL 11.26 — PUBLICATION ARTIFACT PERSISTENCE
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


PUBLICATION_FILES = []

for root in [
    TABLE_DIR,
    FIGURE_DIR,
    SUMMARY_DIR,
    STATISTICS_DIR
]:

    if not root.exists():
        continue

    for path in root.rglob("*"):

        if path.is_file():

            PUBLICATION_FILES.append({

                "file":
                    str(
                        path.relative_to(
                            PUBLICATION_DIR
                        )
                    ),

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(path)

            })


PUBLICATION_MANIFEST_DF = pd.DataFrame(
    PUBLICATION_FILES
)


PUBLICATION_MANIFEST_DF.to_csv(
    MANIFEST_DIR
    / "publication_artifact_manifest.csv",
    index=False
)


print("=" * 100)
print("PUBLICATION ARTIFACT PERSISTENCE")
print("=" * 100)

print(
    f"Files persisted : "
    f"{len(PUBLICATION_MANIFEST_DF)}"
)

display(
    PUBLICATION_MANIFEST_DF.head(20)
)

In [ ]:
# ============================================================
# CELL 11.27 — NOTEBOOK 11 MANIFEST
# ============================================================

NOTEBOOK_11_MANIFEST = {

    "notebook":
        "11_Final_Publication_Results",

    "framework":
        "AIR-LLM",

    "purpose":
        "Final publication-ready aggregation, "
        "statistical reporting, tables, figures, "
        "and reproducibility validation",

    "datasets":
        DATASETS,

    "target_registry":
        TARGET_REGISTRY,

    "mechanisms":
        MECHANISMS,

    "missingness_rates":
        MISSINGNESS_RATES,

    "repetitions":
        REPETITIONS,

    "master_seed":
        MASTER_SEED,

    "input_notebooks": [
        "03_Missingness_Generation",
        "04_Dataset_and_Feature_Profiling",
        "05_LLM_Recommendation_Engine",
        "06_Candidate_Strategy_Evaluation",
        "07_Adaptive_Utility_Based_Selection",
        "08_Confidence_Abstention",
        "09_Explainability",
        "10_Cross_Dataset_Robustness_Statistical_Analysis"
    ],

    "output_directory":
        str(PUBLICATION_DIR),

    "tables_directory":
        str(TABLE_DIR),

    "figures_directory":
        str(FIGURE_DIR),

    "statistics_directory":
        str(STATISTICS_DIR),

    "summary_directory":
        str(SUMMARY_DIR),

    "artifact_count":
        len(PUBLICATION_FILES),

    "validation":
        VALIDATION,

    "generated_utc":
        RUN_TIMESTAMP
}


NOTEBOOK_11_MANIFEST_PATH = (
    MANIFEST_DIR
    / "notebook_11_manifest.json"
)


with open(
    NOTEBOOK_11_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        NOTEBOOK_11_MANIFEST,
        f,
        indent=2,
        default=str
    )


print(
    f"Manifest saved:\n"
    f"{NOTEBOOK_11_MANIFEST_PATH}"
)

In [ ]:
# ============================================================
# CELL 11.28 — FINAL NOTEBOOK STATUS
# ============================================================

ALL_VALIDATIONS_PASSED = all(
    VALIDATION.values()
)


print("\n" + "=" * 100)
print("AIR-LLM — NOTEBOOK 11 COMPLETE")
print("=" * 100)

print("\nFINAL RESEARCH OUTPUT")
print("-" * 100)

print(
    f"Datasets                    : "
    f"{len(DATASETS)}"
)

print(
    f"Selected strategy records   : "
    f"{len(FINAL_SELECTION_DF):,}"
)

print(
    f"Unique strategies selected  : "
    f"{SUMMARY['unique_selected_strategies']}"
)

print(
    f"Statistical comparisons     : "
    f"{len(STATISTICAL_DF):,}"
)

print(
    f"Effect-size comparisons     : "
    f"{len(EFFECT_SIZE_DF):,}"
)

print(
    f"Publication tables          : "
    f"{len(table_files)}"
)

print(
    f"Publication figures         : "
    f"{len(figure_files)}"
)

print(
    f"Publication artifacts       : "
    f"{len(PUBLICATION_FILES)}"
)


print("\nVALIDATION")
print("-" * 100)

for key, value in VALIDATION.items():

    print(
        f"{key:35s}: "
        f"{'PASSED' if value else 'FAILED'}"
    )


print("\nOUTPUT")
print("-" * 100)

print(
    f"Publication directory:\n"
    f"{PUBLICATION_DIR}"
)

print(
    f"Tables:\n"
    f"{TABLE_DIR}"
)

print(
    f"Figures:\n"
    f"{FIGURE_DIR}"
)

print(
    f"Statistics:\n"
    f"{STATISTICS_DIR}"
)

print(
    f"Summary:\n"
    f"{SUMMARY_PATH}"
)

print(
    f"Manifest:\n"
    f"{NOTEBOOK_11_MANIFEST_PATH}"
)


print("\n" + "=" * 100)

if ALL_VALIDATIONS_PASSED:

    print(
        "ALL NOTEBOOK 11 VALIDATIONS PASSED"
    )

    print(
        "AIR-LLM FINAL PUBLICATION RESULTS ARE READY"
    )

    print(
        "THE EXPERIMENTAL PIPELINE IS COMPLETE"
    )

else:

    print(
        "NOTEBOOK 11 COMPLETED WITH VALIDATION WARNINGS"
    )

    print(
        "Review the validation results before manuscript preparation."
    )

print("=" * 100)

gc.collect()